### 시각-언어-행동 통합 Agent (모델 기반 반사 에이전트)
- InputImage : 이미지를 읽는 역할
- VisionAnalysis : 비전 모델을 활용하여 이미지를 분석
- CaptionExtract : 언어 모델을 활용하여 이미지를 설명
- ActionExecute : 다음 행동을 선택하고 실행하는 역할

In [1]:
# bitsandbytes : 양자화 라이브러리
# pyngrok : 터널링 도구
!pip install -qU ultralytics openai bitsandbytes streamlit pyngrok

# qwen_vl_utils : Qwen 모델에서 이미지·비디오 전처리 및 입력 변환을 돕는 유틸리티 모듈
# av : Python 바인딩을 제공하는 FFmpeg 래퍼 (비디오/오디오 파일 디코딩 & 인코딩,
#      프레임 단위 접근, 다양한 코덱(H.264, VP9 등) 지원, FFmpeg과 동일한 강력한 포맷 호환성)
!pip install -q qwen_vl_utils av

!pip install -q diffusers

!pip install -qU langchain>=1.0.0 langchain-community langchain-experimental langchain-text-splitters langchain_tavily
!pip install -qU langchain-openai langchain-google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 69.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 62.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 130.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 145.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.9/68.9 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 75.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
     ━━━━━━━━━━━━

In [2]:
import matplotlib.pyplot as plt
import cv2, base64, time, os, torch, io, json, re
from PIL import Image
import numpy as np
from tqdm import tqdm
import base64

from ultralytics import YOLO
from openai import OpenAI
# import google.generativeai as genai # 구버전 SDK
# 신버전 SDK
from google import genai
from google.genai import types

# AI 모델을 웹 데모로 손쉽게 만드는 라이브러리
import gradio as gr
from pyngrok import ngrok
from pathlib import Path

from transformers import CLIPProcessor, CLIPModel
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration
# 양자화
from transformers import BitsAndBytesConfig

# 모델에 대한 비디오 데이터 준비
from qwen_vl_utils import process_vision_info
from diffusers import StableDiffusionPipeline
from diffusers import StableDiffusion3Pipeline

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.


In [3]:
# 라이브러리 import
from uuid import uuid4

# 타입 설정
# Sequence : 메시지들의 순서 있는 시퀀스(리스트) 객체 (list 또는 tuple 형태로 메시지를 저장)
# Literal : 값 자체를 타입으로 제한할 수 있도록 해주는 기능
from typing import Annotated, TypedDict, Dict, List, Sequence, Literal, Any
# pydantic : Python 데이터 검증과 설정 관리를 위한 라이브러리
from pydantic import Field, BaseModel, ValidationError

# 상태, 노드
# 노드 : 그래프에서의 작업 단위, 실제 작업을 수행하는 함수
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, START, END
# 도구를 노드로 만드는 기능 (자동 맵핑, 상태업데이트, 다중 실행)
from langgraph.prebuilt import ToolNode
# 도구 노드의 결과에 따라 자동으로 라우팅 경로를 설정
from langgraph.prebuilt import tools_condition

# 메모리 저장소
from langgraph.checkpoint.memory import MemorySaver

# 실행 설정
from langchain_core.runnables import RunnableConfig

# 시각화
from IPython.display import Image, display, Markdown

from langchain.tools import tool # 에이전트가 스스로 선택하는 도구
# 모델 연동
import openai
from langchain.chat_models import init_chat_model
from langchain_openai.embeddings import OpenAIEmbeddings

# 프롬프트
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# 메모리 기능
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage, SystemMessage

from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

/tmp/ipykernel_3675/1707933318.py:40: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.chat_message_histories import ChatMessageHistory


In [4]:
# API key 설정
import os

In [5]:
os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY"
os.environ["GOOGLE_API_KEY"] = "YOUR_GOOGLE_API_KEY"

#### 상태정의

In [ ]:
class VisionAgentState(TypedDict) :
  image_path : Annotated[str, "input image path"]
  origin_image : Annotated[str, "origin image data (base64 encoded)"]
  model_version : Annotated[str, "Ultralytics YOLO model name"]
  detections : Annotated[List[Dict[str,str]], "YOLO detection results"]
  caption_text : Annotated[str, "Multi modal LLM : predict results"]
  request : Annotated[str, "user's query"]
  final_msg : Annotated[str, "final message"]

#### 노드정의

In [ ]:
# 이미지 입력 노드
def InputImage(s:VisionAgentState) -> VisionAgentState :
  # 이미지 오픈
  img = cv2.imread(s['image_path'])

  if img is None : # 이미지가 오픈되지 않는 경우
    print("Log : 이미지 오픈에 문제가 있습니다.")
    return VisionAgentState(origin_image = "None")
  else : # 이미지가 오픈되는 경우 base64로 직렬화 (인코딩)
    _, buffer = cv2.imencode(".jpg", img)
    base64_encoded = base64.b64encode(buffer)
    base64_string = base64_encoded.decode('utf-8')
    # print("직렬화 결과 : ", base64_string)
    return VisionAgentState(origin_image = base64_string)

In [ ]:
# 이미지 분석 노드
def VisionAnalysis(s:VisionAgentState) -> VisionAgentState :
  # 모델생성
  model = YOLO(s['model_version'])
  # 모델추론
  ## base64 -> numpy image data
  if s['origin_image'] != "None" :
    img_bytes = base64.b64decode(s['origin_image'])
    nparr = np.frombuffer(img_bytes, np.uint8)
    img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
  results = model(img)
  # print(results[0].boxes)
  # 예측정보 추출(정리) -> 클래스 이름 / 확신도 / 객체의 위치(바운딩박스 좌표)
  detections = [] # 추론결과를 담을 리스트
  for box in results[0].boxes :
    # 검출객체의 이름 확보
    label = model.names[int(box.cls[0].item())]
    # 검출객체에 대한 확신도 추출
    conf = float(box.conf[0].item())
    # 객체의 위치정보 추출(시작 x, 시작 y, 끝 x, 끝 y)
    x1,y1,x2,y2 = box.xyxy[0]
    # 추출된 결과를 리스트에 저장
    detections.append({
        "label" : label, "confidence" : f"{conf:.2f}",
        "x1" : str(int(x1)), "y1" : str(int(y1)),
        "x2" : str(int(x2)), "y2" : str(int(y2))
    })

  return VisionAgentState(detections = detections)

In [ ]:
# 이미지 설명 노드
def CaptionExtract(s:VisionAgentState) -> VisionAgentState :
  prompt = f'''
    당신은 사용자의 영양상태를 케어하는 agent입니다.
    아래 객체 탐지결과(detection results)를 참고하여
    영양상태에 대한 설명 caption text를 구성하세요. (음식의 종류, 크기, 칼로리 등)
    caption text는 5줄 이내로 작성하세요.

    detection results : {s['detections']}
  '''
  # OpenAI모델 생성 및 멀티모달 추론 호출
  client = OpenAI()
  response = client.responses.create(
      model="gpt-5.4-mini", # 사용할 모델이름
      input = [
          {
              'role' : "user",
              'content' : [
                  {'type' : "input_text", "text" : prompt},
                  {'type' : "input_image", "image_url" : f"data:image/jpg;base64,{s['origin_image']}"}
              ]
          }
      ]
  )
  return VisionAgentState(caption_text = response.output_text) # 응답결과를 상태에 업데이트

In [ ]:
# 웹 검색 도구
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_tavily import TavilySearch
# 웹 검색을 위한 Key 등록
os.environ['TAVILY_API_KEY'] = 'YOUR_TAVILY_API_KEY'
# 검색도구 생성
tool_tavily = TavilySearch(max_results=3)

In [ ]:
# tool 정의
@tool
def web_search(msg : str) -> str :
  '''
    매개변수로 들어온 msg를 기반으로 운동방법과 관련된 웹 검색을 수행하고 검색결과를 반환
  '''
  rs = tool_tavily.invoke(msg)
  rs_results = rs['results'] # 검색결과만 추출
  text_results = [ {'source' : doc['url'], 'content' : doc['content'] } for doc in rs_results ]
  print("[웹 검색 수행]")
  return json.dumps(text_results,ensure_ascii=False )

In [ ]:
@tool
def send_message(msg : str) -> str :
  '''
    매개변수로 들어온 msg를 사용자가 지정한 곳으로 전송
  '''
  print("[발송 msg] : ", msg)
  return "msg를 발송했습니다."

In [ ]:
# LLM binding
llm = init_chat_model(model="openai:gpt-4o-mini")
llm_with_tools = llm.bind_tools([web_search, send_message])
# llm_with_tools.invoke("3주간 다이어트를 하기위한 운동법 알려줘")
llm_with_tools.invoke("3주간 다이어트 식단정보를 내 메일로 보내줘.")

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 45, 'prompt_tokens': 113, 'total_tokens': 158, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_cdd4b75452', 'id': 'chatcmpl-ELfVSniU3EQKwMJeeebDB0kzZcgQV', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a07ec3-a5ba-7183-a867-d05a1265f9dc-0', tool_calls=[{'name': 'send_message', 'args': {'msg': '3주간 다이어트 식단 정보를 요청하셨습니다. 제가 빅데이터와 웹 검색을 통해 적절한 정보를 찾아보겠습니다.'}, 'id': 'call_yKAMbv0T7hQInFITzye6DHqf', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 113, 'output_tokens':

In [ ]:
# 도구 모음
tool_by_name = {"web_search" : web_search, "send_message" : send_message}

In [ ]:
# 액션 노드 구성(메세지 송신, 웹 검색, 하드웨어 제어 등)
def ActionExecute(s:VisionAgentState) -> VisionAgentState :
  system_prompt = '''
    당신은 사용자의 식이요법, 운동법 등을 관리하는 agent 입니다.
    AI가 분석한 식단 정보(Caption)과 사용자의 요구사항(request)를
    분석해 웹검색/메세지 송신 등 필요한 tool이 있다면 호출하고 결과를 반환하세요.

    Caption : {caption}
  '''
  template = ChatPromptTemplate.from_messages([
      ('system', system_prompt),
      ('human', "request : {request}")
  ])
  chain = template | llm_with_tools
  # 메세지를 담을 리스트
  messages = [SystemMessage(content=ChatPromptTemplate.from_template(system_prompt).invoke(s['caption_text']).messages[0].content),
              HumanMessage(content=f"request : {s['request']}")]
  response = chain.invoke({"caption" : s['caption_text'], "request" : s['request'] })
  messages.append(response) # 응답메세지 추가
  if response.tool_calls : # 만약 tool call이 발생하는 경우
    for tc in response.tool_calls :
      # 이름에 맞는 도구 가져오기
      selected_tool = tool_by_name[tc['name']]
      # 도구 실행
      output = selected_tool.invoke(tc['args'])
      messages.append(ToolMessage(content=output, tool_call_id = tc['id']))
    # 최종응답 얻기
    final_response = llm_with_tools.invoke(messages)
    return VisionAgentState(final_msg = final_response.content)
  else : # tool call이 발생하지 않는 경우
    return VisionAgentState(final_msg = response.content)

#### workflow 구성 및 실행

In [ ]:
# 그래프 베이스 생성
workflow = StateGraph(VisionAgentState)
# 노드 추가
workflow.add_node("InputImage", InputImage)
workflow.add_node("VisionAnalysis",VisionAnalysis)
workflow.add_node("CaptionExtract",CaptionExtract)
workflow.add_node('ActionExecute',ActionExecute)
# 엣지 추가
workflow.add_edge(START, "InputImage")
workflow.add_edge("InputImage","VisionAnalysis")
workflow.add_edge("VisionAnalysis","CaptionExtract")
workflow.add_edge("CaptionExtract","ActionExecute")
workflow.add_edge("ActionExecute", END)
# 컴파일
app = workflow.compile()

In [ ]:
for rs in app.stream(VisionAgentState(image_path="/content/food.jpg",
                                 model_version="yolo26n.pt",
                                 request="사진의 영양상태를 분석하고 어울리는 3주간 운동법 검색해줘")) :
  print(rs)

{'InputImage': {'origin_image': '...'}}

0: 416x640 1 fork, 1 knife, 2 bowls, 2 oranges, 2 carrots, 1 dining table, 186.0ms
Speed: 9.2ms preprocess, 186.0ms inference, 1.7ms postprocess per image at shape (1, 3, 416, 640)
{'VisionAnalysis': {'detections': [{'label': 'dining table', 'confidence': '0.80', 'x1': '0', 'y1': '1', 'x2': '480', 'y2': '304'}, {'label': 'bowl', 'confidence': '0.80', 'x1': '30', 'y1': '58', 'x2': '253', 'y2': '287'}, {'label': 'bowl', 'confidence': '0.80', 'x1': '269', 'y1': '97', 'x2': '446', 'y2': '276'}, {'label': 'orange', 'confidence': '0.73', 'x1': '360', 'y1': '108', 'x2': '413', 'y2': '190'}, {'label': 'knife', 'confidence': '0.55', 'x1': '110', 'y1': '19', 'x2': '352', 'y2': '52'}, {'label': 'orange', 'confidence': '0.50', 'x1': '396', 'y1': '97', 'x2': '444', 'y2': '185'}, {'label': 'carrot', 'confidence': '0.47', 'x1': '161', 'y1': '174', 'x2': '224', 'y2': '262'}, {'label': 'fork', 'confidence': '0.33', 'x1': '115', 'y1': '20', 'x2': '354', 'y2': '51

In [ ]:
rs = app.invoke(VisionAgentState(image_path="/content/food.jpg",
                                 model_version="yolo26n.pt",
                                 request="사진의 영양상태를 분석하고 결과 메일로 보내줘"))


0: 416x640 1 fork, 1 knife, 2 bowls, 2 oranges, 2 carrots, 1 dining table, 292.3ms
Speed: 33.5ms preprocess, 292.3ms inference, 1.3ms postprocess per image at shape (1, 3, 416, 640)
[발송 msg] :  영양 상태 분석 결과:

- 샐러드 및 구운 닭가슴살 : 250~350kcal (단백질, 채소 포함)
- 과일 및 견과류 : 150~250kcal (과일, 건강한 지방 포함)

전체 칼로리: 400~600kcal

식사구성이 균형잡혀 있으며, 단백질, 채소, 건강한 지방, 과일이 포함되어 있습니다.
[웹 검색 수행]


In [ ]:
rs['final_msg']

'영양 상태 분석 결과를 이메일로 발송했습니다. \n\n분석 내용은 다음과 같습니다:\n- 샐러드 및 구운 닭가슴살: 250~350kcal (단백질, 채소 포함)\n- 과일 및 견과류: 150~250kcal (과일, 건강한 지방 포함)\n- 전체 칼로리: 400~600kcal\n\n이 식사는 균형잡힌 구성을 가지고 있으며, 단백질, 채소, 건강한 지방과 과일이 포함되어 있습니다.\n\n추가로 영양 상태 분석을 이메일로 보내는 방법에 대한 정보를 찾았습니다:\n1. [영양 상태 분석 입력 및 분석 방법](https://www.kns.or.kr/can6.0/CanPro6n_result.asp) - 섭취 음식 영양소 정보, 24시간 회상법 결과물 제공\n2. [바이오컴 검체 반송 방법](https://biocom.kr/qna) - 영양 검사 관련 정보\n3. [영양판정 정의 및 조사 방법](https://lib.gnu.ac.kr#/search/detail/608380) - 여러 조사 방법에 대한 설명\n\n이 정보들이 도움이 되길 바랍니다!'

### 비디오 분석 Vision Agent

#### 학습 내용
- CLIP를 활용한 비디오 유해성 판단
- Gemini를 활용한 비디오 유해성 근거 설명
- Qwen2.5-VL을 활용한 정상 비디오 요약
- 엔드투엔드 솔루션에 필요한 감지 → 설명 → 요약 파이프라인 구축

<center>  
<img src="https://arome1004.cafe24.com/images/cv_project/lecture_image/video_understand.png" width=50%>
</center>

#### CLIP (Contrastive Language-Image Pre-training)
- **Contrastive Learning**
  - 레이블링 없이 학습하는 비지도 학습(Self-supervised Learning)
  - 특정 입력을 Embedding Network를 통해 임베딩 공간으로 이동시키고, 같은 class라면 임베딩 값의 거리를 최소화(d+)하고, 다른 class라면 임베딩 값의 거리를 최대화 (d-) 되도록 embedding network를 학습


<center>  
<img src="https://arome1004.cafe24.com/images/cv_project/lecture_image/clip01.png" width=40%> <br><font size=1>출처 : https://xoft.tistory.com/67</font>  
</center>

- CLIP 모델은 2021년에 OpenAI에서 개발한 모델
- 4억개 이미지와 해당 이미지에 대한 설명 Text를 pair로 두고 학습데이터셋으로 구성한 후, 각각을 인코더로 임베딩하여 같은 pair에 대해 거리를 가깝게하고 다른 pair에 대해 거리가 멀어지도록 텍스트/이미지 인코더를 학습


<center>  
<img src="https://arome1004.cafe24.com/images/cv_project/lecture_image/clip02.png" width=40%> <br><font size=1>출처 : https://openai.com/index/clip/</font>  
</center>

##### CLIP 모델로딩

In [6]:
clip_checkpoint = "openai/clip-vit-base-patch32"
model_clip = CLIPModel.from_pretrained(clip_checkpoint)
processor_clip = CLIPProcessor.from_pretrained(clip_checkpoint)
# GPU 자원이 확보된 경우 처리
clip_device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("CLIP device : ", clip_device)
model_clip = model_clip.to(clip_device)

config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  605MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  605MB            

...

vocab.json:   0%|          | 0.00/862k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.22M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

CLIP device :  cuda


##### Qwen2.5-VL 모델 다운로드

In [7]:
if torch.cuda.is_available() : # GPU사용이 가능한 경우
  quant_config = BitsAndBytesConfig(
    load_in_4bit=True, # 모델저장을 16비트를 4비트로 줄임 -> 약간의 성능저하, 속도 향상
    bnb_4bit_compute_dtype=torch.float16, # 연산(추론)은 잠깐 16bit로 복원해서 진행
    bnb_4bit_use_double_quant=True, # 양자화 활성화
    bnb_4bit_quant_type="nf4", # 정규분포 형태로 노말라이즈
  )
  # 모델 다운로드
  model_qwen = Qwen2_5_VLForConditionalGeneration.from_pretrained(
      'Qwen/Qwen2.5-VL-3B-Instruct',
      quantization_config = quant_config
  )
else : # cpu환경인 경우
  model_qwen = Qwen2_5_VLForConditionalGeneration.from_pretrained(
      'Qwen/Qwen2.5-VL-3B-Instruct'
  )

processor_qwen = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-3B-Instruct")
model_qwen.eval() # 추론모드 설정

config.json:   0%|          | 0.00/1.37k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/65.4k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/5.70k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Qwen2_5_VLForConditionalGeneration(
  (model): Qwen2_5_VLModel(
    (visual): Qwen2_5_VisionTransformerPretrainedModel(
      (patch_embed): Qwen2_5_VisionPatchEmbed(
        (proj): Conv3d(3, 1280, kernel_size=(2, 14, 14), stride=(2, 14, 14), bias=False)
      )
      (rotary_pos_emb): Qwen2_5_VisionRotaryEmbedding()
      (blocks): ModuleList(
        (0-31): 32 x Qwen2_5_VLVisionBlock(
          (norm1): Qwen2_5_VLRMSNorm((1280,), eps=1e-06)
          (norm2): Qwen2_5_VLRMSNorm((1280,), eps=1e-06)
          (attn): Qwen2_5_VLVisionAttention(
            (qkv): Linear4bit(in_features=1280, out_features=3840, bias=True)
            (proj): Linear4bit(in_features=1280, out_features=1280, bias=True)
          )
          (mlp): Qwen2_5_VLMLP(
            (gate_proj): Linear4bit(in_features=1280, out_features=3420, bias=True)
            (up_proj): Linear4bit(in_features=1280, out_features=3420, bias=True)
            (down_proj): Linear4bit(in_features=3420, out_features=1280, bias=True

#### Vision function 정의

In [149]:
# 프레임 추출용 함수
def extract_frames(video_path : str, step : int = 10) -> list[tuple[int,str]] :
  ''' 비디오에서 전체 프레임을 추출하는 함수 '''
  cap = cv2.VideoCapture(video_path)
  if not cap.isOpened() :
    print('[Log] 비디오 파일을 열 수 없습니다.')
  else :
    print('[Log] 비디오 파일 오픈 성공')
    frames = [] # 프레임을 담을 리스트
    idx = 0 # 프레임 번호를 관리할 변수

    while True :
      ret, frame = cap.read() # 비디오 프레임 읽기
      if ret == False : # 더이상 읽을 프레임이 없는 경우
        break
      if idx % step == 0 : # 일정 주기마다 샘플링 되도록 설정
        # 색공간 변환
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        # base64로 직렬화
        _, buffer = cv2.imencode(".jpg", rgb_frame)
        base64_encoded = base64.b64encode(buffer)
        base64_string = base64_encoded.decode('utf-8')
        # 리스트에 추가
        frames.append( (idx,base64_string) )
      idx += 1
    cap.release() # 스트림 닫기
    return frames

In [156]:
rs = extract_frames("/content/face_1s.mp4",step=5)

[Log] 비디오 파일 오픈 성공


In [157]:
rs

[(0,
  '...'),
 (5,
  '...'),
 (10,
  '...'),
 (15,
  '...'),
 (20,
  '...'),
 (25,
  '...')]

In [170]:
# 입력 : base64로 인코딩된 사진데이터
# 출력 : 유해성 클래스와 확신도(conf)
def classify_frame_with_clip(frame : str) -> tuple[str,float] :
  ''' CLIP으로 단일 프레임의 유해성 라벨과 신뢰도를 측정 '''
  # 입력된 프레임과 유사도 비교를 할 문장
  text_inputs = [
      "이 이미지는 정상입니다.",
      "이 이미지에는 누드가 포함되어 있습니다.",
      "이 이미지에는 유혹적이거나 감각적인 내용이 포함되어 있습니다.",
      "이 이미지에는 폭력적인 내용이 포함되어 있습니다."
  ]
  # 추론 결과를 표현할 카테고리 클래스
  labels = ['정상', '누드', '감각적', '폭력적']

  # 전처리
  ## base64 to imge
  img_bytes = base64.b64decode(frame)
  nparr = np.frombuffer(img_bytes, np.uint8)
  img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)

  inputs = processor_clip(
      text=text_inputs,
      images=img,
      return_tensors='pt',
      padding=True
  ).to(clip_device)
  # 모델추론
  with torch.no_grad():
    outputs = model_clip(**inputs)
  # 다중분류 확률값으로 변경
  probs = outputs.logits_per_image.softmax(dim=-1)
  # 제일 높은 확률 값과 클래스 번호 추출
  conf, pred_class = probs.max(dim=-1)
  return float(conf.item()), labels[pred_class.item()]

In [171]:
classify_frame_with_clip(rs[0][1])

(0.48535865545272827, '감각적')

In [173]:
def analyze_video_safety(
    sampled_frames:list[tuple[int,str]],
    threshold:float = 0.25
) -> tuple[str,float,list[dict[str,Any]],list[str]] :
  ''' 샘플링된 프레임을 검사하고 유해 프레임 목록을 반환 '''
  flagged_data = [] # 유해성 검사 결과 데이터 리스트
  flagged_frames = [] # 유해성 검사 결과 프레임 리스트

  for frame_id, b64str in tqdm(sampled_frames) : # 샘플링된 프레임 반복처리
    conf, label = classify_frame_with_clip(b64str) # 한 프레임 유해성 검증
    if label != '정상' and conf >= threshold :
      flagged_data.append({
          'frame_id' : frame_id, # 프레임 번호
          'classification' : label, # 분류결과
          'confidence' : conf # 확신도(신뢰도)
      })
      flagged_frames.append(b64str) # 프레임 추가
  # 영상에 대한 종합 판단 라벨링
  safety_label = 'unsafe' if flagged_frames else 'safe'
  # 유해한 프레임 비율계산
  unsafe_ratio = len(flagged_frames) / len(sampled_frames) * 100
  # 결과반환
  return safety_label,unsafe_ratio,flagged_data,flagged_frames

In [178]:
result = analyze_video_safety(rs, threshold=0.25)



  0%|          | 0/6 [00:00<?, ?it/s]

 17%|█▋        | 1/6 [00:01<00:05,  1.07s/it]

 33%|███▎      | 2/6 [00:02<00:04,  1.07s/it]

 50%|█████     | 3/6 [00:04<00:04,  1.48s/it]

 67%|██████▋   | 4/6 [00:05<00:02,  1.42s/it]

 83%|████████▎ | 5/6 [00:06<00:01,  1.30s/it]

100%|██████████| 6/6 [00:07<00:00,  1.26s/it]


In [179]:
result

('unsafe',
 100.0,
 [{'frame_id': 0, 'classification': '감각적', 'confidence': 0.48535865545272827},
  {'frame_id': 5, 'classification': '감각적', 'confidence': 0.414902925491333},
  {'frame_id': 10, 'classification': '감각적', 'confidence': 0.3654949963092804},
  {'frame_id': 15, 'classification': '감각적', 'confidence': 0.3662973940372467},
  {'frame_id': 20, 'classification': '감각적', 'confidence': 0.4295578598976135},
  {'frame_id': 25, 'classification': '감각적', 'confidence': 0.4032135605812073}],
 ['...'])

In [188]:
# gemini를 활용한 유해성 설명 함수
client = genai.Client(api_key=os.environ['GOOGLE_API_KEY'])
# base64 -> 바이트이미지
def base64_to_image(base64_str) :
  ''' base64 이미지데이터를 gemini가 인식하는 데이터형태로 감싸서 반환'''
  img_bytes = base64.b64decode(base64_str)
  return types.Part.from_bytes(
      data = img_bytes,
      mime_type = "image/jpeg"
  )
def explain_flagged_frames_gemini(
    flagged_data : list[dict[str,Any]],
    flagged_frames : list[str]
) :
  ''' 유해성을 설명하도록 요청하는 함수 '''

  prompt = f'''
    다음은 비디오에서 유해할 수 있다고 분류된 프레임 정보입니다.

    {json.dumps(flagged_data, ensure_ascii=False)}

    각 프레임의 이미지를 함께 보고 다음 내용을 2~3문장으로 설명하세요.

    1. 어떤 장면으로 보이는지
    2. 왜 유해 콘텐츠로 판단되었을 가능성이 있는지
    3. 오탐 가능성 또는 모델 판단의 한계

    주의:
      - 단정적으로 말하지 마세요.
      - 이미지에서 보이는 정보만 근거로 설명하세요.
      - 확실하지 않은 경우 '추정'이라고 표현하세요.
  '''
  # 제미나이에 전송할 데이터 리스트
  content = []
  content.append(prompt) # 유해성 메타데이터가 포함된 프롬프트 추가
  for f in flagged_frames : # 유해성 이미지 반복처리
    content.append( base64_to_image(f) ) # base64 to bytes로 변경 후 리스트에 추가

  # 호출
  response = client.models.generate_content(
      model="gemini-3.6-flash",
      contents = content
  )
  return response.text.strip()

In [187]:
explain_flagged_frames_gemini(result[2], result[3])

ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

In [189]:
def explain_flagged_frames_openai(
    flagged_data : list[dict[str,Any]],
    flagged_frames : list[str]
) :
  ''' 유해성을 설명하도록 요청하는 함수 '''

  prompt = f'''
    다음은 비디오에서 유해할 수 있다고 분류된 프레임 정보입니다.

    {json.dumps(flagged_data, ensure_ascii=False)}

    각 프레임의 이미지를 함께 보고 다음 내용을 2~3문장으로 설명하세요.

    1. 어떤 장면으로 보이는지
    2. 왜 유해 콘텐츠로 판단되었을 가능성이 있는지
    3. 오탐 가능성 또는 모델 판단의 한계

    주의:
      - 단정적으로 말하지 마세요.
      - 이미지에서 보이는 정보만 근거로 설명하세요.
      - 확실하지 않은 경우 '추정'이라고 표현하세요.
  '''
  # OpenAI모델 생성 및 멀티모달 추론 호출
  client = OpenAI()
  response = client.responses.create(
      model="gpt-5.4-mini", # 사용할 모델이름
      input = [
          {
              'role' : "user",
              'content' : [
                  {'type' : "input_text", "text" : prompt},
                  {'type' : "input_image", "image_url" : f"data:image/jpg;base64,{flagged_frames[0]}"}
              ]
          }
      ]
  )

  return response.output_text

In [190]:
explain_flagged_frames_openai(result[2], result[3])

'이 프레임은 두 사람이 가까이 서서 카메라를 바라보며 미소 짓는 장면으로 보입니다. 조명과 구도가 다소 연출된 분위기이고, 피부 노출이나 친밀한 포즈가 있어 모델이 감각적인 장면으로 추정했을 가능성이 있습니다. 다만 실제로는 일반적인 인물 사진일 수 있어 오탐 가능성이 있으며, 단일 프레임만으로는 유해성 판단에 한계가 있습니다.'

In [8]:
def summarize_video_qwen(video_path) :
  '''정상 영상인 경우 Qwen2.5-VL로 비디오 요약'''
  # Qwen 모델을 위한 video path 타입으로 생성
  vpath = Path(video_path).resolve()
  # 요청할 메세지 구성
  messages = [
      {
       "role" : "user",
       "content" : [
           {'type':"video", "video" : f"file://{vpath}"},
           {"type": "text", "text": (
                "이 영상을 한글로 요약하세요. "
                "반드시 JSON 형식으로 반환하세요. "
                "키는 bullets와 paragraph만 사용하세요. "
                "bullets는 5개 이내의 문자열 리스트, paragraph는 짧은 문단입니다."
           )}
       ]
      }
  ]
  # 전처리 작업 실시
  text = processor_qwen.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
  )
  image_inputs, video_inputs = process_vision_info(messages)

  inputs = processor_qwen(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        return_tensors="pt",
  ).to(model_qwen.device)

  # 모델추론
  with torch.no_grad():
    output_ids = model_qwen.generate(**inputs, max_new_tokens=512)

  trimmed = [
        out_ids[len(in_ids):]
        for in_ids, out_ids in zip(inputs.input_ids, output_ids)
  ]
  # 추론한 토큰숫자 값을 글자로 복원하여 반환
  return processor_qwen.batch_decode(trimmed, skip_special_tokens=True)[0].strip()

In [10]:
 rs = summarize_video_qwen("/content/face_1s.mp4")

[transformers] Qwen2VL video processing does not apply the per-frame pixel cap the reference implementation (qwen-vl-utils) applies, so some videos cost far more tokens than they would there. In v5.22 the capped behavior will become the default and `cap_pixels_per_frame` will be removed. Pass `cap_pixels_per_frame=True` to adopt the reference behavior now, or `False` to keep the current behavior and silence this warning.


In [11]:
rs

'{\n  "bullets": [\n    "Two women standing close together",\n    "One woman has blonde hair and is wearing a pink top",\n    "The other woman has long brown hair and is wearing a red top",\n    "They both have big smiles on their faces",\n    "The background is a warm yellow color"\n  ],\n  "paragraphs": [\n    "The two women are posing for the camera.",\n    "Their smiles suggest they are happy or excited about something.",\n    "The warm yellow background gives the video a cheerful vibe.",\n    "The women\'s outfits contrast nicely with each other.",\n    "This image could be from a music video or a promotional video."\n  ]\n}'